# Session 1: Clinical NLP with OpenAI GPT API

---

이 튜토리얼에서는 OpenAI의 **Responses API**를 활용하여 임상노트(Clinical Note)에서 유용한 정보를 추출하는 방법을 실습합니다.

### Learning Objectives

| Task | Description |
|------|-------------|
| **Task 1** | Drug Name Extraction — 약물명 및 ATC 코드 추출 |
| **Task 2** | Adverse Drug Reaction Detection — 약물 부작용 탐지 및 검증 |
| **Task 3** | Clinical Note Summarization — 임상노트 한국어 요약 |

### Requirements

- `openai` >= 1.0
- OpenAI API Key

---
## 0. Environment Setup

In [1]:
from openai import OpenAI
import pandas as pd

# ── Configuration ──────────────────────────────────────────────
PATH = "./DATA/"
API_KEY = "" # "" 안에 OpenAI GPT API key를 복사해서 붙여넣기를 진행합니다.

In [ ]:
# ── Download sample clinical notes ─────────────────────────────
BASE_URL = "https://raw.githubusercontent.com/leelabsg/clinical-llm-tutorial/main/Session1"

!wget -qP $PATH {BASE_URL}/note1_ShorthandStyle.txt
!wget -qP $PATH {BASE_URL}/note2_MIMICstyle.txt

print("Download complete.")

---
## Clinical Note Examples

본 실습에서는 두 가지 스타일의 임상노트를 제공합니다.

| Note | Style | Description |
|------|-------|-------------|
| `note1` | **Shorthand** | 간단한 요약형 노트 (약어, 목록 중심) |
| `note2` | **Narrative (MIMIC-style)** | 서술형 노트 (상세한 문맥과 진단 경과 포함) |

두 노트를 각각 사용하여 prompt 차이, 언어 차이, 노트 형식 차이에 따른 분석 결과를 비교해볼 수 있습니다.

### Note 1 — Shorthand Style

In [ ]:
with open(f"{PATH}note1_ShorthandStyle.txt", "r", encoding="utf-8") as f:
    note1 = f.read()

print(note1)

### Note 2 — Narrative Style (MIMIC-style)

In [ ]:
with open(f"{PATH}note2_MIMICstyle.txt", "r", encoding="utf-8") as f:
    note2 = f.read()

print(note2[:500])  # 앞 500자만 출력

### Select Working Note

본 실습에서는 `note1` (shorthand version)을 사용합니다.  
Narrative version을 테스트하려면 아래를 `note = note2`로 변경하세요.

In [ ]:
note = note1

---
## 1. Initialize OpenAI Client

In [ ]:
client = OpenAI(api_key=API_KEY)

---
## Task 1: Drug Name Extraction & Drug Code Identification

임상노트로부터 언급된 약물명을 추출하는 작업을 수행합니다.

- GPT에게 임상노트를 입력하고, 텍스트에서 **약물명** 추출 및 약물에 상응하는 **ATC CODE**를 추출합니다.
- 약물은 **일반명** 또는 **약어**(예: MFM = metformin)로 표현될 수 있습니다.

### 1-1. Drug Name Only

In [ ]:
prompt1_eng = f"""
Task: Extract all drug names mentioned in the following clinical note.
- Drug names may be written as abbreviations.
- Return the list of drug names only, without any additional explanation.

Clinical Note:
{note}
"""

In [ ]:
MODEL = "gpt-5.6"

response = client.responses.create(
    model=MODEL,
    instructions="You are a clinical text extraction assistant.",
    input=prompt1_eng,
)

print(response.output_text)

### 1-2. Drug Name + Description + ATC Code

In [ ]:
prompt2_eng = f"""
Task: Extract all drug names mentioned in the following clinical note.
- Drug names may be written as abbreviations.
- Return the list of drug names, simple descriptions, and corresponding ATC codes in a table format (CSV).

Clinical Note:
{note}
"""

response = client.responses.create(
    model=MODEL,
    instructions="You are a clinical text extraction assistant.",
    input=prompt2_eng,
)

print(response.output_text)

### 1-3. Usage Check

사용한 토큰 수를 확인할 수 있습니다.

In [ ]:
print(response.usage)

---
## Task 2: Adverse Drug Reaction (ADR) Detection

임상노트에 언급된 **약물 부작용(Adverse Drug Reactions, ADR)** 을 탐지하는 작업을 수행합니다.

- GPT에게 임상노트를 입력하고, 특정 약물 투여 이후 발생한 부작용을 **약물-증상 쌍(drug → symptom)** 형태로 추출합니다.
- 약물명은 약어로 표현될 수 있으며, 부작용은 명시적 또는 암시적으로 표현되어 있을 수 있습니다.

### 2-1. ADR Extraction

In [ ]:
prompt_adr = f"""
Task: Extract all possible adverse drug reactions (ADRs) mentioned in the clinical note.
- Return results in JSON format, with each item containing:
  - "drug": drug name
  - "symptom": symptom or adverse effect
  - "evidence": supporting sentence or phrase from the note
- No extra explanation.

Clinical Note:
{note}
"""

response = client.responses.create(
    model=MODEL,
    instructions="You are a clinical text extraction assistant.",
    input=prompt_adr,
)

print(response.output_text)

### 2-2. Verification Agent

LLM이 생성한 ADR 추출 결과가 타당한지 검증하기 위해, **별도의 Verifier Agent**를 생성하여 답변을 교차 검증합니다.

In [ ]:
verifier_prompt = """
If you disagree, provide counterarguments.
Provide agree and comment for each drug and side effect pairs.
Respond in JSON:
{"agree": true/false, "comments": "..."}.
"""

extractor_output = response.output_text

verifier_input = (
    f"{verifier_prompt}\n\n"
    f"ExtractorAgent output:\n{extractor_output}\n\n"
    f"Medical note:\n{note}"
)

VERIFIER_MODEL = "gpt-5.6-mini"  # lightweight model for verification

verifier_response = client.responses.create(
    model=VERIFIER_MODEL,
    instructions="You are VerifierAgent. Your job is to check ExtractorAgent's identified side effects.",
    input=verifier_input,
)

print(verifier_response.output_text)

---
## Task 3: Clinical Note Summarization

임상노트에 포함된 핵심 정보를 GPT를 통해 한글로 **요약(summarization)** 하는 작업을 수행합니다.

- 임상노트에는 환자의 병력, 진단, 투약, 검사 결과, 치료 계획 등의 정보가 포함되어 있으며, 종종 매우 길고 복잡합니다.
- 이 Task는 환자에 대한 중요한 정보를 요약하여, 환자/보호자 설명에 활용될 수 있는 간결한 요약문을 생성하는 것이 목적입니다.
- 출력은 한국어 요약문으로 생성됩니다.

In [ ]:
prompt_family = f"""
Task: Summarize the following clinical note in simple language for the patient's family.
- Use non-technical, plain Korean language.
- The summary should explain what happened, what was diagnosed, how it was treated, and what to expect.
- Avoid complex terminology and abbreviations.

Output language: Korean

Clinical Note:
{note2}
"""

In [ ]:
response = client.responses.create(
    model=MODEL,
    instructions="You are a clinical text summarization assistant.",
    input=prompt_family,
)

print(response.output_text)

---

## Summary

| Task | Method | Key Takeaway |
|------|--------|------|
| **Drug Extraction** | Prompt engineering | 약어 포함 약물명 + ATC 코드 추출 가능 |
| **ADR Detection** | Extractor + Verifier | Multi-agent 패턴으로 결과 신뢰도 향상 |
| **Summarization** | Korean plain-language | 환자/보호자용 비전문 요약 생성 |